In [4]:
import requests
from bs4 import BeautifulSoup
import csv
import time

def scrape_reviews(domain: str):
    all_reviews = []

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/122.0.0.0 Safari/537.36"
        ),
        "Accept-Language": "en-US,en;q=0.9",
    }

    session = requests.Session()
    session.headers.update(headers)

    for page_num in range(1, 6):
        url = f"https://www.trustpilot.com/review/{domain}?page={page_num}"
        print(f"Scraping page {page_num}: {url}")

        resp = session.get(url, timeout=25)
        print(f"  HTTP status: {resp.status_code}, bytes: {len(resp.text)}")

        if resp.status_code != 200:
            print(f"  Skipped (HTTP {resp.status_code})")
            time.sleep(2)
            continue

        soup = BeautifulSoup(resp.text, "html.parser")

        # Trustpilot reviews are usually in <article>
        review_elements = soup.find_all("article")
        print(f"  Found articles: {len(review_elements)}")

        for item in review_elements:
            try:
                # Reviewer name
                name_tag = item.find("span", {"data-consumer-name-typography": True})
                name = name_tag.get_text(strip=True) if name_tag else "Unknown"

                # Review date
                date = "Unknown"
                time_tag = item.find("time")
                if time_tag and time_tag.has_attr("datetime"):
                    date = time_tag["datetime"].split("T")[0]

                # Title + body
                title_tag = item.find("h2", {"data-service-review-title-typography": True})
                title = title_tag.get_text(strip=True) if title_tag else ""

                body_tag = item.find("p", {"data-review-content-typography": True})
                body = body_tag.get_text(strip=True) if body_tag else ""

                content = f"{title}: {body}".strip().strip(": ").strip()

                if not content:
                    continue

                all_reviews.append({
                    "Reviewer name": name,
                    "Review date": date,
                    "Review content": content
                })
            except Exception:
                continue

        time.sleep(2)

    return all_reviews


def save_to_csv(data_list, filename: str):
    fieldnames = ["Reviewer name", "Review date", "Review content"]

    with open(filename, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(data_list)

    print(f"Saved: {filename}")
    print(f"Total reviews saved: {len(data_list)}")


if __name__ == "__main__":
    # Watsons that actually has Trustpilot reviews:
    target_domain = "www.watsons.com.sg"
    data = scrape_reviews(target_domain)
    save_to_csv(data, "scraped_reviews.csv")

Scraping page 1: https://www.trustpilot.com/review/www.watsons.com.sg?page=1
  HTTP status: 200, bytes: 705827
  Found articles: 24
Scraping page 2: https://www.trustpilot.com/review/www.watsons.com.sg?page=2
  HTTP status: 200, bytes: 668835
  Found articles: 24
Scraping page 3: https://www.trustpilot.com/review/www.watsons.com.sg?page=3
  HTTP status: 200, bytes: 614740
  Found articles: 24
Scraping page 4: https://www.trustpilot.com/review/www.watsons.com.sg?page=4
  HTTP status: 200, bytes: 627473
  Found articles: 24
Scraping page 5: https://www.trustpilot.com/review/www.watsons.com.sg?page=5
  HTTP status: 200, bytes: 605791
  Found articles: 24
Saved: scraped_reviews.csv
Total reviews saved: 100


Muhammad Haziq bin Abdullah - SW01093756
Mohamad Naqib bin Mustapa - SW01083743